# 3. Baselines and BERT training (Tasks 1 to 3)

| | |
|---|---|
| **Stage** | 3 of 9 |
| **Purpose** | Train and compare TF-IDF baselines and BERT for overall review sentiment (Task 1), sentence-level aspect detection (Task 2) and aspect sentiment (Task 3); run the Task 2 cross-validation, choose the deployed threshold, and save the final models. |
| **Inputs** | `data/processed/{game}_clean.csv (with overall_sentiment_weak)`<br>`data/annotation_v2/gold_set_combined.csv`<br>`scripts/aspect_taxonomy.py` |
| **Outputs** | `models/task1_bert_combined`<br>`models/task2_bert_gold, models/task2_final`<br>`models/task3_bert_gold, models/task3_final`<br>`outputs/baseline_summary_v2.json, outputs/bert_summary_v2.json, outputs/deploy_config.json`<br>`outputs/oof_*.npy` |
| **Runtime** | GPU required (Colab T4). |
| **Depends on** | Stages 1 and 2 |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

> **Note.** The Task 1 and Task 2 final models used by the final pipeline come from this notebook. The Task 3 models trained here are the first version and are superseded by notebooks 04 and 07.

#### Step 1: Access Google Drive and Gold Set

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = '/content/drive/MyDrive/GameSense'

# ---------- create folders ----------
REQUIRED_DIRS = [
    f'{PROJECT_ROOT}/data',
    f'{PROJECT_ROOT}/data/annotation_v2',
    f'{PROJECT_ROOT}/data/processed',
    f'{PROJECT_ROOT}/scripts',
    f'{PROJECT_ROOT}/models',
    f'{PROJECT_ROOT}/models/task1_bert_combined',
    f'{PROJECT_ROOT}/models/task2_bert_gold',
    f'{PROJECT_ROOT}/models/task3_bert_gold',
    f'{PROJECT_ROOT}/outputs',
]
for d in REQUIRED_DIRS:
    os.makedirs(d, exist_ok=True)
print("Done✅")

# ---------- check necessnary docs ----------
REQUIRED_FILES = [
    f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv',
    f'{PROJECT_ROOT}/scripts/aspect_taxonomy.py',
] + [f'{PROJECT_ROOT}/data/processed/{slug}_clean.csv'
     for slug in ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]]

missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
if missing:
    print("⚠️ Mandatory docs missing：")
    for f in missing:
        print("   ", f)
else:
    print("Pass ✅")

# ---------- reach gold set ----------
gold_df = pd.read_csv(f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv')
print(f'successfully!: {len(gold_df)} sentences, cover {gold_df["game"].nunique()} games')
gold_df.head(3)

#### Step 2: Loading Taxonomy

In [ ]:
import sys, importlib
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
if 'aspect_taxonomy' in sys.modules:
    importlib.reload(sys.modules['aspect_taxonomy'])
from aspect_taxonomy import ASPECT_TAXONOMY, ASPECT_IDS

print(f'Taxonomy loading successful，total {len(ASPECT_IDS)} categories')

#### Step 3: None-DNN Baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

Y_cols = [f'gold_{a}' for a in ASPECT_IDS]
for col in Y_cols:
    gold_df[col] = pd.to_numeric(gold_df[col], errors='coerce').fillna(0).astype(int)

Y = gold_df[Y_cols].values
X_text = gold_df['sentence_text'].fillna('')

idx = np.arange(len(gold_df))
idx_train, idx_temp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=5000, sublinear_tf=True)
Xtr = vec.fit_transform(X_text.iloc[idx_train])
Xval = vec.transform(X_text.iloc[idx_val])
Xte = vec.transform(X_text.iloc[idx_test])

clf = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight='balanced'))
clf.fit(Xtr, Y[idx_train])

pred_test = clf.predict(Xte)
print("=== Real Gold Baseline ===")
print(f"total samples in test set: {len(idx_test)}")
print(f"test set macro-F1: {f1_score(Y[idx_test], pred_test, average='macro', zero_division=0):.3f}")
print(f"test set micro-F1: {f1_score(Y[idx_test], pred_test, average='micro', zero_division=0):.3f}")

print(f"\nAll category's F1 (support=The actual number of sentences of this category in the test set):")
per_class_f1 = f1_score(Y[idx_test], pred_test, average=None, zero_division=0)
for aspect_id, f1 in zip(ASPECT_IDS, per_class_f1):
    support = int(Y[idx_test][:, ASPECT_IDS.index(aspect_id)].sum())
    print(f"  {aspect_id:30s} F1={f1:.3f}  (support={support})")

#### Step 4: Data preparation (expanding into sentence-aspect pairs)

In [ ]:
def explode_to_pairs(df):
    rows = []
    for _, row in df.iterrows():
        for a in ASPECT_IDS:
            if row[f'gold_{a}'] == 1:
                sentiment = row.get(f'gold_{a}_sentiment', '')
                if pd.isna(sentiment) or sentiment == '':
                    continue
                rows.append({
                    'sentence_text': row['sentence_text'],
                    'aspect_id': a,
                    'aspect_label': ASPECT_TAXONOMY[a]['label'],
                    'sentiment': sentiment,
                })
    return pd.DataFrame(rows)

pairs_df = explode_to_pairs(gold_df)
print(f"Total {len(pairs_df)} (sentence, aspect) pairs.")
print(pairs_df['sentiment'].value_counts())

#### Step 5: Save Baseline result summary

In [ ]:
import json

summary = {
    "gold_set_size": len(gold_df),
    "games_covered": int(gold_df['game'].nunique()),
    "task2_test_macro_f1": round(f1_score(Y[idx_test], pred_test, average='macro', zero_division=0), 3),
    "task2_test_micro_f1": round(f1_score(Y[idx_test], pred_test, average='micro', zero_division=0), 3),
    "task3_total_pairs": len(pairs_df),
}
print(summary)

with open(f'{PROJECT_ROOT}/outputs/baseline_summary_v2.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Saved baseline_summary_v2.json")

#### Step 6: BERT Preparation

6a. Installing dependencies, imports, and common utilities

In [ ]:
%pip install -q -U transformers accelerate

import os, shutil, time, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
    DataCollatorWithPadding, set_seed,
)

print("GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected. Please go to Runtime → Change runtime type → T4 GPU, then re-run from Cell 1."
print("GPU Type:", torch.cuda.get_device_name(0))

MODEL_NAME = "bert-base-uncased"
set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)   # Dynamic padding: Each batch is padded only to the length of the longest sequence within that batch, which is much faster than padding everything to a fixed length of 256.

# ---------- Common dataset (shared by three tasks） ----------
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item

# ---------- Weighted Trainer for single-label classification (used for Task 1 and Task 3) ----------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# ---------- Weighted Trainer for Multi-label Classification (for Task 2) ----------
class WeightedMultiLabelTrainer(Trainer):
    def __init__(self, pos_weight=None, **kwargs):
        super().__init__(**kwargs)
        self.pos_weight = pos_weight
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_3class(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

print("Done ✅")

6b. Merge reviews from five games (for Task 1)

In [ ]:
GAME_SLUGS = ["royalmatch", "genshin", "mlbb", "pubgm", "arknights"]
all_reviews = []
for slug in GAME_SLUGS:
    df = pd.read_csv(f'{PROJECT_ROOT}/data/processed/{slug}_clean.csv')
    df['game'] = slug
    all_reviews.append(df)

reviews_df = pd.concat(all_reviews, ignore_index=True)
reviews_df = reviews_df.dropna(subset=["content_clean", "overall_sentiment_weak"])
print(f"After merge total {len(reviews_df)} comments, cover {reviews_df['game'].nunique()} games")
print(reviews_df['overall_sentiment_weak'].value_counts())

#### Step 7: Task 1 - Overall Sentiment Classification

7a. Data preparation

In [ ]:
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

reviews_df["label_id"] = reviews_df["overall_sentiment_weak"].map(LABEL2ID)

train_r, temp_r = train_test_split(reviews_df, test_size=0.30, random_state=42, stratify=reviews_df["label_id"])
val_r, test_r = train_test_split(temp_r, test_size=0.50, random_state=42, stratify=temp_r["label_id"])

def enc_review(df):
    return tokenizer(list(df["content_clean"]), truncation=True, max_length=256)

train_ds1 = TextDataset(enc_review(train_r), train_r["label_id"].tolist())
val_ds1 = TextDataset(enc_review(val_r), val_r["label_id"].tolist())
test_ds1 = TextDataset(enc_review(test_r), test_r["label_id"].tolist())

class_weights1 = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(3), y=train_r["label_id"]), dtype=torch.float
)
print("Done ✅", len(train_ds1), len(val_ds1), len(test_ds1))

7b. Building the Model and Trainer

In [ ]:
TASK1_LOCAL_DIR = '/content/task1_tmp'
TASK1_MODEL_DIR = f'{PROJECT_ROOT}/models/task1_bert_combined'
if os.path.exists(TASK1_LOCAL_DIR):
    shutil.rmtree(TASK1_LOCAL_DIR)
os.makedirs(TASK1_MODEL_DIR, exist_ok=True)

model1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

args1 = TrainingArguments(
    output_dir=TASK1_LOCAL_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,                    # If a NaN loss occurs, delete this line.
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer1 = WeightedTrainer(
    class_weights=class_weights1,
    model=model1, args=args1,
    train_dataset=train_ds1, eval_dataset=val_ds1,
    data_collator=collator,
    compute_metrics=compute_metrics_3class,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("Done✅")

7c. Training

In [ ]:
t0 = time.time()
trainer1.train()
print(f"\nTask1 training time: {(time.time() - t0) / 60:.1f} mins")

7d. Evaluate and save

In [ ]:
test_pred1 = trainer1.predict(test_ds1)
preds1 = np.argmax(test_pred1.predictions, axis=-1)
print("=== Task 1 BERT Test set results ===")
print(classification_report(test_r["label_id"], preds1,
                            target_names=["negative","neutral","positive"], zero_division=0))
task1_macro_f1 = f1_score(test_r["label_id"], preds1, average="macro", zero_division=0)
print(f"Task1 test set macro-F1: {task1_macro_f1:.3f}")

trainer1.save_model(TASK1_MODEL_DIR)
tokenizer.save_pretrained(TASK1_MODEL_DIR)
print(f"Task1 model saved -> {TASK1_MODEL_DIR}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec1 = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=50000, sublinear_tf=True)
Xtr1 = vec1.fit_transform(train_r["content_clean"])
Xte1 = vec1.transform(test_r["content_clean"])
clf1 = LogisticRegression(max_iter=2000, class_weight="balanced")
clf1.fit(Xtr1, train_r["label_id"])
p1 = clf1.predict(Xte1)
print("=== Task 1 TF-IDF Baseline（Same test set as BERT）===")
print(classification_report(test_r["label_id"], p1, target_names=["negative","neutral","positive"], zero_division=0))
print(f"macro-F1: {f1_score(test_r['label_id'], p1, average='macro'):.3f}   (BERT: 0.664)")

**When a problem occurs?**


1.   Encountering NaN or OOM errors during training, or wanting to retrain with different hyperparameters. -> Re-run *7b* and *7c*.
2.   After training, I want to change the threshold and review the results. -> Modify the THRESHOLD in step *7a*, re-run that line, and then run step *7d* (no need to retrain the model).
3. An error occurred only during evaluation or saving. -> Re-run *7d*.
4. Runtime disconnection or restart. -> Re-run *all notebook cells*.

**Task 1: Overall sentiment classification**


*What it does*: classifies each full review as negative, neutral or positive. It gives the dashboard its headline view of how players feel about each game.


*Labels*: they are weak, derived from the star rating. 1–2 stars is negative, 3 is neutral, 4–5 is positive. That gives 18,358 positive, 13,928 negative and only 1,691 neutral (about 5%) across 33,977 reviews. Your proposal (Section 5.2) restricts star-derived labels to this task and never applies them to aspects.


*Model*: bert-base-uncased with a 3-way classification head on the [CLS] representation. Max length 256 tokens, because reviews are long. Weighted cross-entropy with "balanced" class weights, so the rare neutral class isn't ignored. Stratified 70/15/15 split, early stopping on validation macro-F1

#### Step 8: Task 2 - Multi-label Aspect Detection

8a. Data preparation

In [ ]:
from sklearn.metrics import average_precision_score

def enc_sent(texts):
    return tokenizer(list(texts), truncation=True, max_length=96)

texts_all = gold_df['sentence_text'].fillna('').tolist()
Y_float = Y.astype(np.float32)

train_ds2 = TextDataset(enc_sent([texts_all[i] for i in idx_train]), Y_float[idx_train].tolist())
val_ds2 = TextDataset(enc_sent([texts_all[i] for i in idx_val]), Y_float[idx_val].tolist())
test_ds2 = TextDataset(enc_sent([texts_all[i] for i in idx_test]), Y_float[idx_test].tolist())

# Weights are square-rooted and capped to avoid over-compensation (previously max 116, now max 6).
pos_counts = Y[idx_train].sum(axis=0)
neg_counts = len(idx_train) - pos_counts
ratio = neg_counts / np.maximum(pos_counts, 1)
pos_weight = torch.tensor(np.clip(np.sqrt(ratio), 1.0, 6.0), dtype=torch.float)
print("pos_weight:", {k: round(v, 1) for k, v in zip(ASPECT_IDS, pos_weight.tolist())})

# During training, the best epoch is selected using threshold-independent macro average precision, eliminating reliance on a fixed threshold.
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    aps = [average_precision_score(labels[:, j], probs[:, j])
           for j in range(labels.shape[1]) if labels[:, j].sum() > 0]
    preds = (probs >= 0.5).astype(int)
    return {
        "macro_ap": float(np.mean(aps)),
        "macro_f1_at_0.5": f1_score(labels, preds, average="macro", zero_division=0),
    }

print("Task2 done ✅", len(train_ds2), len(val_ds2), len(test_ds2))

8b. Building the Model and Trainer

In [ ]:
import math

TASK2_LOCAL_DIR = '/content/task2_tmp'
TASK2_MODEL_DIR = f'{PROJECT_ROOT}/models/task2_bert_gold'
if os.path.exists(TASK2_LOCAL_DIR):
    shutil.rmtree(TASK2_LOCAL_DIR)
os.makedirs(TASK2_MODEL_DIR, exist_ok=True)

NUM_EPOCHS = 30
BATCH = 16
total_steps = math.ceil(len(train_ds2) / BATCH) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)      # Use the first 10% of the steps as a warm-up.
print(f"Total Steps: {total_steps}, warmup steps: {warmup_steps}")

model2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(ASPECT_IDS),
    problem_type="multi_label_classification",
    id2label={i: a for i, a in enumerate(ASPECT_IDS)},
    label2id={a: i for i, a in enumerate(ASPECT_IDS)},
)

args2 = TrainingArguments(
    output_dir=TASK2_LOCAL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=warmup_steps,        # replace warmup_ratio
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_ap",
    greater_is_better=True,
    logging_steps=10,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer2 = WeightedMultiLabelTrainer(
    pos_weight=pos_weight,
    model=model2, args=args2,
    train_dataset=train_ds2, eval_dataset=val_ds2,
    data_collator=collator,
    compute_metrics=compute_metrics_multilabel,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)
print("Task2 done ✅")

8c. Training

In [ ]:
t0 = time.time()
trainer2.train()
print(f"\nTask2 training time: {(time.time() - t0) / 60:.1f} mins")

In [ ]:
import pandas as pd
logs = pd.DataFrame(trainer2.state.log_history)

train_loss = logs[logs['loss'].notna()][['epoch', 'loss']].groupby(logs['epoch'].round()).last()
ev = logs[logs['eval_loss'].notna()][['epoch', 'eval_loss', 'eval_macro_ap', 'eval_macro_f1_at_0.5']]
print(ev.to_string(index=False))
best = ev.loc[ev['eval_macro_ap'].idxmax()]
print(f"\nthe best epoch（rank by macro_ap）: {int(best['epoch'])}，macro_ap={best['eval_macro_ap']:.3f}")
print(f"final training loss: {logs[logs['loss'].notna()]['loss'].iloc[-1]:.3f}")

In [ ]:
from sklearn.metrics import average_precision_score

def macro_ap(y_true, y_prob):
    aps = [average_precision_score(y_true[:, j], y_prob[:, j])
           for j in range(y_true.shape[1]) if y_true[:, j].sum() > 0]
    return float(np.mean(aps))

bert_ap = macro_ap(Y[idx_test], test_probs)
base_ap = macro_ap(Y[idx_test], clf.predict_proba(Xte))

print("=== Test set macro-average precision (threshold-independent) ===")
print(f"TF-IDF baseline: {base_ap:.3f}")
print(f"BERT:            {bert_ap:.3f}")

8d. Evaluation and save

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 1) Perform a threshold sweep on the validation set. Select the threshold based on micro-F1, as the validation set contains only 75 sentences, making macro-F1 too volatile.
val_probs = sigmoid(trainer2.predict(val_ds2).predictions)
best_t, best_f1 = 0.5, -1
for t in np.arange(0.05, 0.96, 0.05):
    f = f1_score(Y[idx_val], (val_probs >= t).astype(int), average="micro", zero_division=0)
    if f > best_f1:
        best_t, best_f1 = float(t), f
THRESHOLD = best_t
print(f"The threshold selected from the validation set: {THRESHOLD:.2f}  (val micro-F1={best_f1:.3f})")

# 2) The test set is evaluated only once using this threshold.
test_probs = sigmoid(trainer2.predict(test_ds2).predictions)
preds2 = (test_probs >= THRESHOLD).astype(int)

task2_macro_f1 = f1_score(Y[idx_test], preds2, average='macro', zero_division=0)
task2_micro_f1 = f1_score(Y[idx_test], preds2, average='micro', zero_division=0)
print("=== Task 2 BERT Test set results ===")
print(f"macro-F1: {task2_macro_f1:.3f}  (baseline: 0.393)")
print(f"micro-F1: {task2_micro_f1:.3f}  (baseline: 0.558)")
print(f"Proportion of positive predictions: {preds2.mean():.3f}   A truly positive proportion: {Y[idx_test].mean():.3f}")

per_class = f1_score(Y[idx_test], preds2, average=None, zero_division=0)
print("\nAll F1 categories:")
for a, f in zip(ASPECT_IDS, per_class):
    print(f"  {a:30s} F1={f:.3f}  (support={int(Y[idx_test][:, ASPECT_IDS.index(a)].sum())})")

trainer2.save_model(TASK2_MODEL_DIR)
tokenizer.save_pretrained(TASK2_MODEL_DIR)

In [ ]:
val_probs_b = clf.predict_proba(Xval)
best_tb, best_fb = 0.5, -1
for t in np.arange(0.05, 0.96, 0.05):
    f = f1_score(Y[idx_val], (val_probs_b >= t).astype(int), average="micro", zero_division=0)
    if f > best_fb:
        best_tb, best_fb = float(t), f

pb = (clf.predict_proba(Xte) >= best_tb).astype(int)
print(f"baseline the threshold selected from the validation set: {best_tb:.2f}")
print(f"baseline(Adjust threshold) macro-F1: {f1_score(Y[idx_test], pb, average='macro', zero_division=0):.3f}")
print(f"baseline(Adjust threshold) micro-F1: {f1_score(Y[idx_test], pb, average='micro', zero_division=0):.3f}")

Error diagnosis

In [ ]:
test_texts = [texts_all[i] for i in idx_test]

def show_errors(aspect_id, max_show=10):
    j = ASPECT_IDS.index(aspect_id)
    y = Y[idx_test][:, j]
    p = test_probs[:, j]
    pred = preds2[:, j]
    print(f"===== {aspect_id}  (Threshold {THRESHOLD:.2f}) =====")
    for k in np.where((y == 1) & (pred == 0))[0][:max_show]:
        print(f"[Missed detection FN] p={p[k]:.2f} | {test_texts[k][:150]}")
    for k in np.where((y == 0) & (pred == 1))[0][:max_show]:
        print(f"[False detection FP] p={p[k]:.2f} | {test_texts[k][:150]}")
    print()

for a in ["character_operator_balance", "content_progression", "grind_progression"]:
    show_errors(a)

In [ ]:
lens = [len(tokenizer(t)["input_ids"]) for t in texts_all]
print("Token length quantiles: 50/90/95/99/max:", np.percentile(lens, [50, 90, 95, 99, 100]))
print("Number of sentences exceeding 96 tokens:", sum(l > 96 for l in lens), "/", len(lens))

j = ASPECT_IDS.index("character_operator_balance")
for k in np.where((Y[idx_test][:, j] == 1) & (preds2[:, j] == 0))[0]:
    t = test_texts[k]
    print(f"\n{len(tokenizer(t)['input_ids'])} tokens | {t}")

In [ ]:
GAME_NAME = {"royalmatch": "Royal Match", "genshin": "Genshin Impact",
             "mlbb": "Mobile Legends", "pubgm": "PUBG Mobile", "arknights": "Arknights"}
texts_game = [f"{GAME_NAME[g]} : {t}"
              for g, t in zip(gold_df["game"], gold_df["sentence_text"].fillna(""))]

**Task 2: Aspect category detection (multi-label)**

*What it does*: decides which of the 12 aspects a sentence mentions. Zero, one or several can apply, so it is multi-label, not multi-class.


*Taxonomy*: 7 common aspects (performance, monetization, advertisement, level difficulty, content and progression, paywall, account and support) and 5 genre-specific ones (gacha rates, character/operator balance, matchmaking, cheating, grind). Genre-specific aspects only apply to certain games, as in your applicability matrix.


*Labels*: your manual gold set of 500 sentences (100 per game), split into roughly 350 train, 75 validation and 75 test. Sentences with no specific issue are all zeros, which is valid.


*Model*: Same BERT encoder, but with a sigmoid output per aspect instead of softmax, so each aspect is an independent yes/no decision. Weighted BCE loss: pos_weight = negatives / positives per aspect. For rare aspects the weight is large (grind_progression is about 116). Decision threshold 0.3 instead of 0.5, so rare aspects can actually fire. Max length 96, since sentences are short

#### Step 9: Task 3 - Aspect-level sentiment classification

9a. Data Preparation

In [ ]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
LABEL2ID_S = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
ID2LABEL_S = {i: l for l, i in LABEL2ID_S.items()}

pairs_df["label_id"] = pairs_df["sentiment"].map(LABEL2ID_S)
pairs_df = pairs_df.dropna(subset=["label_id"]).copy()
pairs_df["label_id"] = pairs_df["label_id"].astype(int)

train_p, temp_p = train_test_split(pairs_df, test_size=0.30, random_state=42, stratify=pairs_df["label_id"])
val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=42, stratify=temp_p["label_id"])

def enc_pair(df):
    return tokenizer(list(df["sentence_text"]), list(df["aspect_label"]), truncation=True, max_length=96)

train_ds3 = TextDataset(enc_pair(train_p), train_p["label_id"].tolist())
val_ds3 = TextDataset(enc_pair(val_p), val_p["label_id"].tolist())
test_ds3 = TextDataset(enc_pair(test_p), test_p["label_id"].tolist())

class_weights3 = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(3), y=train_p["label_id"]), dtype=torch.float
)
print("Task3 Category weights:", dict(zip(SENTIMENT_LABELS, [round(w, 2) for w in class_weights3.tolist()])))
print("Task3 Done ✅", len(train_ds3), len(val_ds3), len(test_ds3))

9b. Building the Model and Trainer

In [ ]:
TASK3_LOCAL_DIR = '/content/task3_tmp'
TASK3_MODEL_DIR = f'{PROJECT_ROOT}/models/task3_bert_gold'
if os.path.exists(TASK3_LOCAL_DIR):
    shutil.rmtree(TASK3_LOCAL_DIR)
os.makedirs(TASK3_MODEL_DIR, exist_ok=True)

model3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL_S, label2id=LABEL2ID_S
)

args3 = TrainingArguments(
    output_dir=TASK3_LOCAL_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

trainer3 = WeightedTrainer(
    class_weights=class_weights3,
    model=model3, args=args3,
    train_dataset=train_ds3, eval_dataset=val_ds3,
    data_collator=collator,
    compute_metrics=compute_metrics_3class,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print("Task3 done ✅")

9c. Training

In [ ]:
t0 = time.time()
trainer3.train()
print(f"\nTask3 training time: {(time.time() - t0) / 60:.1f} mins")

9d. Evaluation and save

In [ ]:
test_pred3 = trainer3.predict(test_ds3)
preds3 = np.argmax(test_pred3.predictions, axis=-1)
print("=== Task 3 BERT Test Set Results ===")
print(classification_report(test_p["label_id"], preds3,
                            target_names=SENTIMENT_LABELS, zero_division=0))
task3_macro_f1 = f1_score(test_p["label_id"], preds3, average="macro", zero_division=0)
print(f"Task3 test set macro-F1: {task3_macro_f1:.3f}")

trainer3.save_model(TASK3_MODEL_DIR)
tokenizer.save_pretrained(TASK3_MODEL_DIR)
print(f"Task3 model saved -> {TASK3_MODEL_DIR}")

In [ ]:
vec3 = TfidfVectorizer(ngram_range=(1,2), min_df=1, sublinear_tf=True)
txt = lambda d: (d["aspect_label"] + " || " + d["sentence_text"]).tolist()
Xtr3 = vec3.fit_transform(txt(train_p)); Xte3 = vec3.transform(txt(test_p))
clf3 = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr3, train_p["label_id"])
p3 = clf3.predict(Xte3)
print(classification_report(test_p["label_id"], p3, target_names=SENTIMENT_LABELS, zero_division=0))
print(f"Task3 baseline macro-F1: {f1_score(test_p['label_id'], p3, average='macro'):.3f}   (BERT: 0.563)")

**Task 3: Aspect-level sentiment classification**

*What it does*: for each aspect present in a sentence, decides whether the sentiment toward that aspect is negative, neutral or positive. In "great game but way too many ads", advertisement is negative and the overall vibe is positive.


*Input format*: sentence-pair classification. The sentence is segment A and the aspect name is segment B, so the model sees [CLS] sentence [SEP] aspect label [SEP]. The aspect label conditions the prediction: the same sentence can get different sentiments for different aspects.


*Labels*: each sentence-aspect pair where you marked the aspect present and gave a sentiment. That yields 417 pairs: 314 negative, 92 positive and only 11 neutral.

*Model*: BERT with a 3-way softmax head, max length 96. Weighted cross-entropy (class weights added to counter the neutral scarcity). Stratified 70/15/15 split (about 291 / 63 / 63 pairs)

#### Step 10: Aggregate and save BERT results

In [ ]:
bert_summary = {
    "task1_overall_sentiment_macro_f1": round(task1_macro_f1, 3),
    "task2_aspect_detection_macro_f1": round(task2_macro_f1, 3),
    "task2_aspect_detection_micro_f1": round(task2_micro_f1, 3),
    "task2_threshold": THRESHOLD,
    "task3_aspect_sentiment_macro_f1": round(task3_macro_f1, 3),
}
print(bert_summary)

with open(f'{PROJECT_ROOT}/outputs/bert_summary_v2.json', 'w') as f:
    json.dump(bert_summary, f, indent=2)
print("Saved: bert_summary_v2.json")

#### CV-1: Fold splitting, baseline, utility functions

In [ ]:
import math, gc, time
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import average_precision_score, f1_score
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def macro_ap_fn(y_true, y_prob):
    aps = [average_precision_score(y_true[:, j], y_prob[:, j])
           for j in range(y_true.shape[1]) if y_true[:, j].sum() > 0]
    return float(np.mean(aps))

GAME_NAME = {"royalmatch": "Royal Match", "genshin": "Genshin Impact",
             "mlbb": "Mobile Legends", "pubgm": "PUBG Mobile", "arknights": "Arknights"}
texts_plain = gold_df["sentence_text"].fillna("").tolist()
texts_game = [f"{GAME_NAME[g]} : {t}" for g, t in zip(gold_df["game"], texts_plain)]
Y_float = Y.astype(np.float32)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(len(gold_df)), gold_df["game"]))
print("Number of positive samples for each category per fold:")
print(pd.DataFrame([Y[te].sum(axis=0) for _, te in folds],
                   columns=ASPECT_IDS, index=[f"fold{i+1}" for i in range(5)]).to_string())

# TF-IDF baseline (same folds, out-of-fold predictions)
oof_base = np.zeros(Y.shape, dtype=float)
for tr, te in folds:
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000, sublinear_tf=True)
    Xa = v.fit_transform([texts_plain[i] for i in tr])
    Xb = v.transform([texts_plain[i] for i in te])
    c = OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xa, Y[tr])
    oof_base[te] = c.predict_proba(Xb)
np.save(f"{PROJECT_ROOT}/outputs/oof_baseline.npy", oof_base)
print(f"\nBaseline 5-fold macro-AP: {macro_ap_fn(Y, oof_base):.3f}")

#### CV-2: BERT Cross-Validation Function

In [ ]:
def run_bert_cv(texts, name, epochs=20, bs=16, lr=5e-5):
    oof = np.zeros(Y.shape, dtype=float)
    for f, (tr, te) in enumerate(folds, 1):
        t0 = time.time()
        set_seed(42 + f)
        enc = lambda idx: tokenizer([texts[i] for i in idx], truncation=True, max_length=96)
        tr_ds = TextDataset(enc(tr), Y_float[tr].tolist())
        te_ds = TextDataset(enc(te), Y_float[te].tolist())

        pc = Y[tr].sum(axis=0)
        nc = len(tr) - pc
        pw = torch.tensor(np.clip(np.sqrt(nc / np.maximum(pc, 1)), 1.0, 6.0), dtype=torch.float)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=len(ASPECT_IDS),
            problem_type="multi_label_classification")
        steps = math.ceil(len(tr) / bs) * epochs
        args = TrainingArguments(
            output_dir=f"/content/cv_tmp_{name}",
            num_train_epochs=epochs, per_device_train_batch_size=bs,
            per_device_eval_batch_size=64, learning_rate=lr,
            warmup_steps=int(0.1 * steps), weight_decay=0.01,
            eval_strategy="no", save_strategy="no", logging_strategy="no",
            fp16=True, report_to="none", disable_tqdm=True, seed=42 + f)
        trainer = WeightedMultiLabelTrainer(
            pos_weight=pw, model=model, args=args,
            train_dataset=tr_ds, data_collator=collator)
        trainer.train()
        oof[te] = sigmoid(trainer.predict(te_ds).predictions)
        print(f"[{name}] fold {f}/5 done，time: {(time.time() - t0) / 60:.1f} mins，"
              f"this fold macro-AP={macro_ap_fn(Y[te], oof[te]):.3f}")
        del trainer, model
        gc.collect(); torch.cuda.empty_cache()

    np.save(f"{PROJECT_ROOT}/outputs/oof_{name}.npy", oof)   # save
    return oof

print("Done ✅")

#### CV-3: Run the "plain" group.

In [ ]:
oof_plain = run_bert_cv(texts_plain, "plain")

#### CV-4: Game-launching prefix group

In [ ]:
oof_game = run_bert_cv(texts_game, "game_prefix")

#### CV-5: Summary Comparison

In [ ]:
cfg = {"TF-IDF baseline": (oof_base, 0.5),
       "BERT plain": (oof_plain, 0.35),
       "BERT game-prefix": (oof_game, 0.35)}

rows = {}
for name, (oof, thr) in cfg.items():
    pr = (oof >= thr).astype(int)
    fold_aps = [macro_ap_fn(Y[te], oof[te]) for _, te in folds]
    rows[name] = {
        "Threshold": thr,
        "macro-AP": round(macro_ap_fn(Y, oof), 3),
        "Inter-run AP (mean ± SD)": f"{np.mean(fold_aps):.3f} ± {np.std(fold_aps):.3f}",
        "macro-F1": round(f1_score(Y, pr, average="macro", zero_division=0), 3),
        "micro-F1": round(f1_score(Y, pr, average="micro", zero_division=0), 3),
        "Predicted to be directly proportional": round(pr.mean(), 3),
    }
print(pd.DataFrame(rows).T.to_string())

per = pd.DataFrame({"support": Y.sum(axis=0)}, index=ASPECT_IDS)
for name, (oof, thr) in cfg.items():
    per[f"{name} AP"] = [average_precision_score(Y[:, j], oof[:, j]) if Y[:, j].sum() > 0 else np.nan
                        for j in range(len(ASPECT_IDS))]
    per[f"{name} F1"] = f1_score(Y, (oof >= thr).astype(int), average=None, zero_division=0)
print("\nAll categories（Based on all）:")
print(per.round(3).to_string())

In [ ]:
COMMON = ["performance_stability", "monetization_value", "advertisement",
          "level_difficulty_balance", "content_progression", "paywall", "account_support"]
GENRE = {"royalmatch": [], "genshin": ["gacha_rates", "character_operator_balance"],
         "mlbb": ["character_operator_balance", "matchmaking", "cheating"],
         "pubgm": ["cheating", "matchmaking"],
         "arknights": ["gacha_rates", "character_operator_balance", "grind_progression"]}

mask = np.zeros(Y.shape, dtype=float)
for i, g in enumerate(gold_df["game"]):
    allowed = set(COMMON) | set(GENRE[g])
    mask[i] = [1.0 if a in allowed else 0.0 for a in ASPECT_IDS]

# Sanity check: gold labels should never fall on non-applicable categories (expected: 0)
print("Number of gold labels falling into not-applicable categories:", int((Y * (1 - mask)).sum()))

def report(name, oof, thr):
    o = oof * mask
    pr = (o >= thr).astype(int)
    print(f"{name:20s} macro-AP={macro_ap_fn(Y, o):.3f}  "
          f"macro-F1={f1_score(Y, pr, average='macro', zero_division=0):.3f}  "
          f"micro-F1={f1_score(Y, pr, average='micro', zero_division=0):.3f}")

print("\n=== With applicability mask (same mask for all three, so still a fair comparison) ===")
report("TF-IDF + mask", oof_base, 0.5)
report("BERT plain + mask", oof_plain, 0.35)
report("BERT prefix + mask", oof_game, 0.35)

print("\n=== Per-fold paired differences (without mask, consistent with the main table) ===")
def fold_aps(oof):
    return np.array([macro_ap_fn(Y[te], oof[te]) for _, te in folds])

for a, b, nm in [(oof_plain, oof_base, "BERT plain - baseline"),
                 (oof_game, oof_plain, "prefix - plain")]:
    d = fold_aps(a) - fold_aps(b)
    print(f"{nm:24s} per fold: {np.round(d, 3)}  mean {d.mean():+.3f}  folds positive {int((d > 0).sum())}/5")

In [ ]:
o = oof_plain * mask
rows = []
for t in np.arange(0.15, 0.61, 0.05):
    pr = (o >= t).astype(int)
    rows.append({"threshold": round(float(t), 2),
                 "macro-F1": f1_score(Y, pr, average="macro", zero_division=0),
                 "micro-F1": f1_score(Y, pr, average="micro", zero_division=0),
                 "pred_pos_rate": pr.mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("true positive rate:", round(float(Y.mean()), 3))

In [ ]:
deploy_cfg = {
    "task2_threshold": 0.30,
    "task2_model_dir": "models/task2_final",
    "task3_model_dir": "models/task3_final",
    "apply_applicability_mask": True,
}
with open(f'{PROJECT_ROOT}/outputs/deploy_config.json', 'w') as f:
    json.dump(deploy_cfg, f, indent=2)
print("Saved deploy_config.json")

In [ ]:
TASK3_FINAL_DIR = f'{PROJECT_ROOT}/models/task3_final'
os.makedirs(TASK3_FINAL_DIR, exist_ok=True)

set_seed(42)
enc3 = tokenizer(list(pairs_df["sentence_text"]), list(pairs_df["aspect_label"]),
                 truncation=True, max_length=96)
full_ds3 = TextDataset(enc3, pairs_df["label_id"].tolist())

cw3 = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(3), y=pairs_df["label_id"]),
    dtype=torch.float)

model3_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL_S, label2id=LABEL2ID_S)

BATCH3, EPOCHS3 = 16, 6      # The validation curve previously peaked between rounds 5 and 7, so round 6 was selected.
args3_final = TrainingArguments(
    output_dir="/content/task3_final_tmp",
    num_train_epochs=EPOCHS3, per_device_train_batch_size=BATCH3,
    learning_rate=2e-5,
    warmup_steps=int(0.1 * math.ceil(len(pairs_df) / BATCH3) * EPOCHS3),
    weight_decay=0.01,
    eval_strategy="no", save_strategy="no", logging_steps=10,
    fp16=True, report_to="none", seed=42,
)
trainer3_final = WeightedTrainer(
    class_weights=cw3, model=model3_final, args=args3_final,
    train_dataset=full_ds3, data_collator=collator,
)

t0 = time.time()
trainer3_final.train()
print(f"training time: {(time.time() - t0) / 60:.1f} mins")

trainer3_final.save_model(TASK3_FINAL_DIR)
tokenizer.save_pretrained(TASK3_FINAL_DIR)
print("Task3 final model saved ->", TASK3_FINAL_DIR)

In [ ]:
import math, time, os
import numpy as np, torch

texts_plain = gold_df["sentence_text"].fillna("").tolist()
Y_float = Y.astype(np.float32)

TASK2_FINAL_DIR = f'{PROJECT_ROOT}/models/task2_final'
os.makedirs(TASK2_FINAL_DIR, exist_ok=True)

set_seed(42)
enc_all = tokenizer(texts_plain, truncation=True, max_length=96)
full_ds = TextDataset(enc_all, Y_float.tolist())

pc = Y.sum(axis=0)
nc = len(Y) - pc
pw_final = torch.tensor(np.clip(np.sqrt(nc / np.maximum(pc, 1)), 1.0, 6.0), dtype=torch.float)

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(ASPECT_IDS),
    problem_type="multi_label_classification",
    id2label={i: a for i, a in enumerate(ASPECT_IDS)},
    label2id={a: i for i, a in enumerate(ASPECT_IDS)},
)

BATCH, EPOCHS = 16, 20
args_final = TrainingArguments(
    output_dir="/content/task2_final_tmp",
    num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
    learning_rate=5e-5,
    warmup_steps=int(0.1 * math.ceil(len(Y) / BATCH) * EPOCHS),
    weight_decay=0.01,
    eval_strategy="no", save_strategy="no", logging_steps=20,
    fp16=True, report_to="none", seed=42,
)
trainer_final = WeightedMultiLabelTrainer(
    pos_weight=pw_final, model=model_final, args=args_final,
    train_dataset=full_ds, data_collator=collator,
)

t0 = time.time()
trainer_final.train()
print(f"training time: {(time.time() - t0) / 60:.1f} mins")

trainer_final.save_model(TASK2_FINAL_DIR)
tokenizer.save_pretrained(TASK2_FINAL_DIR)
print("Final model saved ->", TASK2_FINAL_DIR)

Saved

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, average_precision_score
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/GameSense'
sys.path.insert(0, f'{PROJECT_ROOT}/scripts')
from aspect_taxonomy import ASPECT_IDS

# ---------- 1. Reconstruct the gold label matrix and folds (using the exact same settings as during training）----------
gold_df = pd.read_csv(f'{PROJECT_ROOT}/data/annotation_v2/gold_set_combined.csv')
Y = np.stack([pd.to_numeric(gold_df[f'gold_{a}'], errors='coerce').fillna(0).astype(int).values
              for a in ASPECT_IDS], axis=1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(len(gold_df)), gold_df['game']))

def macro_ap_fn(y_true, y_prob):
    aps = [average_precision_score(y_true[:, j], y_prob[:, j])
           for j in range(y_true.shape[1]) if y_true[:, j].sum() > 0]
    return float(np.mean(aps))

# ---------- 2. Load saved OOF predictions ----------
oofs = {name: np.load(f'{PROJECT_ROOT}/outputs/oof_{name}.npy')
        for name in ['baseline', 'plain', 'game_prefix']}
for k, v in oofs.items():
    assert v.shape == Y.shape, f"{k} Shape mismatch: {v.shape} vs {Y.shape}"

def cv_row(oof, thr):
    pr = (oof >= thr).astype(int)
    fold_ap = [macro_ap_fn(Y[te], oof[te]) for _, te in folds]
    return {"threshold": thr,
            "macro_ap": round(macro_ap_fn(Y, oof), 3),
            "macro_ap_fold_mean": round(float(np.mean(fold_ap)), 3),
            "macro_ap_fold_sd": round(float(np.std(fold_ap)), 3),
            "macro_f1": round(f1_score(Y, pr, average="macro", zero_division=0), 3),
            "micro_f1": round(f1_score(Y, pr, average="micro", zero_division=0), 3)}

cv = {"tfidf_baseline": cv_row(oofs['baseline'], 0.5),
      "bert_plain": cv_row(oofs['plain'], 0.35),
      "bert_game_prefix": cv_row(oofs['game_prefix'], 0.35)}

# ---------- 3. Reproducibility check: Must be consistent with previously printed results ----------
expected = {"tfidf_baseline": (0.518, 0.456), "bert_plain": (0.644, 0.618),
            "bert_game_prefix": (0.658, 0.648)}
for k, (ap, f1) in expected.items():
    ok = abs(cv[k]["macro_ap"] - ap) < 0.002 and abs(cv[k]["macro_f1"] - f1) < 0.002
    print(f"{k:18s} AP={cv[k]['macro_ap']}  F1={cv[k]['macro_f1']}  {'✅ Consistent with the previous version' if ok else '⚠️ Please let me know if this is inconsistent with the previous information'}")

# ---------- 4. Assemble and save (the single-point figure is taken from the printout generated during training) ----------
with open(f'{PROJECT_ROOT}/outputs/deploy_config.json') as f:
    deploy_thr = json.load(f)["task2_threshold"]

summary = {
    "task1_overall_sentiment": {
        "test_macro_f1_bert": 0.664, "test_macro_f1_tfidf": 0.631,
        "note": "star-derived weak labels; n_test=5097; recorded from training run"},
    "task2_aspect_detection": {
        "primary_5fold_cv": cv,
        "single_split_test": {"n_test": 75, "threshold": 0.35,
                              "macro_f1": 0.669, "micro_f1": 0.726,
                              "note": "n=75, high variance; recorded from training run"},
        "deployed_threshold": deploy_thr},
    "task3_aspect_sentiment": {
        "v1_gold_only_test_macro_f1": 0.563,
        "note": "v1: n_test=63, neutral support=1; v2 results to be added"},
}
with open(f'{PROJECT_ROOT}/outputs/bert_summary_v2.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))